In [145]:
import ast
import glob
import json
import math
import os
import pathlib
import onnxruntime as ort

import numpy as np
import pandas as pd
import seaborn as sns
import torch
import tqdm
from matplotlib import pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import linregress
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score

from lib.datasets.base_dataset import gender_mapping, ethnicity_mapping
from lib.hpo_tree.hpo_model import HumanPhenotypeModel, create_model
from lib.hpo_tree.hpo_term import HumanPhenotypeTerm
from lib.utils.mediapipe_helper import extract_face_meshes

plt.style.use('default')
matplotlib.use('pdf')

In [146]:
output_dir = 'X:/hellmafa/facemeshsyndromes/output'
data_dir = 'data'
publication_dir = 'publication'
metric = 'auroc'

metric_mean = f'{metric}_mean'
metric_std = f'{metric}_std'
os.makedirs(publication_dir, exist_ok=True)

In [147]:
df_hpo_annotation = pd.read_csv(os.path.join(data_dir, 'Face2HPO-181120251320.csv'))
df_gmdb = pd.read_csv(os.path.join(data_dir, 'gmdb_data_v110.csv'), low_memory=False)

images_annotated = set(df_hpo_annotation['image_id'].tolist())
images_available = set(df_gmdb['image_id'].tolist())

print(f'Images from the hpo annotation dataset overlapping with the GMDB dataset: {len(images_available.intersection(images_annotated))}')
print(f'Images not in GMDB: {sorted(images_annotated - images_available)}')

df_merged = pd.read_csv('data/gmdb_hpo_facemesh_data_v110.csv')
df_reduced = df_merged[df_merged['image_id'].isin(images_annotated)]
print(f'Number of train/val images: ', len(df_reduced))
print(f'Patients with multiple images: {df_reduced[df_reduced['patient_id'].duplicated()]['patient_id'].unique().tolist()}')
train_val_disorders = df_reduced['internal_syndrome_name'].unique().tolist()
train_val_disorders

Images from the hpo annotation dataset overlapping with the GMDB dataset: 1304
Images not in GMDB: [55, 5938, 6444, 6445, 6447, 8934, 8938, 10421, 11629, 11632, 11633, 11634, 12528, 13470, 13741, 14135, 14182, 14395, 14414, 14492, 14577, 14611, 14639, 14677, 15271, 15275, 15313, 15807, 15809, 15821, 15822, 15823, 15824, 15826, 15830, 15832, 15833, 15834, 15836, 15837, 15838, 15839, 15843, 15844, 15845, 15846, 15903, 15904, 15919, 15943, 16019, 16020, 16024, 16025, 16026, 16035, 16239, 16247, 16423, 16488, 16566, 16588, 16687, 16797, 16802, 16923, 18608, 19217, 19221, 19229, 19237, 19291, 19292, 19293, 19295, 19296, 19297, 19299, 19300, 19301, 19308, 19309, 19310, 19311, 19338, 19361, 19479, 19485, 19486, 19490, 19526, 19550, 19551, 19552, 19642, 19709, 19710, 19712, 19716, 19718, 19719, 19721, 19723, 19725, 19727, 19729, 19730, 19731, 19732, 19734, 19738, 19739, 19741, 19845, 19847, 20107, 20119, 20121, 20124, 20126, 20127, 20128, 20129, 20135, 20137, 20138, 20139, 20143, 20226, 20228,

['WILLIAMS-BEUREN SYNDROME; WBS',
 'Cornelia de Lange syndrome',
 'Hypoplastic left heart syndrome',
 'Kabuki syndrome',
 'Noonan syndrome',
 'Noonan syndrome-like disorder with loose anagen hair',
 'KBG SYNDROME; KBGS',
 'Coffin-Siris syndrome',
 'SCHAAF-YANG SYNDROME; SHFYNG',
 'Mucopolysaccharidoses',
 'Hyperphosphatasia with mental retardation syndrome',
 'OGDEN SYNDROME; OGDNS',
 'ANGELMAN SYNDROME; AS',
 'Waardenburg syndrome']

In [148]:
hpo = HumanPhenotypeTerm.load_ontology(data_dir, download=False)
hp_abnorm_face = hpo.find_successor('HP:0000271')  # Abnormality of the face
hp_abnorm_eye = hpo.find_successor('HP:0000478')  # Abnormality of the eye
hp_abnorm_eyebrow = hpo.find_successor('HP:0000534')  # Abnormal eyebrow morphology
hp_abnorm_face.add_successor(hp_abnorm_eye)  # Move eye to face
hp_abnorm_face.add_successor(hp_abnorm_eyebrow)  # Move eyebrow to face
hp_abnorm_face.define_as_root()
hpo = hp_abnorm_face

keep_hpos = [410030, 160, 2714, 233, 219, 232, 10803, 347, 278, 303, 307, 430, 9928, 9931, 463, 455, 437, 12810, 446,
             5280, 3196, 2000, 322, 11829, 275, 12368, 325, 4428, 1999, 280, 4493, 11800, 10669, 293, 294, 290, 341,
             348, 2007, 45075, 2223, 2553, 664, 336, 11231, 12745, 494, 508, 581, 286, 100539, 486, 525, 568, 601, 520,
             1010, 369, 154, 10805, 12471, 179, 431, 3189, 343, 289, 9890, 337, 574, 637, 582, 316]
keep_hpos = [f'HP:{h:07d}' for h in keep_hpos]
print(f'Annotated HPO-terms: {len(keep_hpos)}')

# Step 2: Find all ancestors of leaves to keep
keep_nodes = set(keep_hpos)
for leaf_id in keep_hpos:
    node = hpo.find_successor(leaf_id)
    while node and node.predecessor:
        keep_nodes.add(node.predecessor.id)
        node = node.predecessor
print(f'Keep HPO-terms: {len(keep_nodes)}')


def prune_subtree(node):
    # Recurse on children first
    node_to_remove = []
    for child in node.successors:
        prune_subtree(child)
        if child.is_leaf() and child.id not in keep_nodes:  # Non-relevant leaf
            node_to_remove.append(child)

    # Remove unnecessary children
    for child in node_to_remove:
        node.remove_successor(child)


prune_subtree(hpo)

hpo_filter = [h.id for h in hpo.all_successors()]
leaf_hpos = [h for h in hpo.all_successors() if h.is_leaf()]
all_hpos = {h.id: h for h in hpo.all_successors()}
print('Amount of HPOs to be used (if trainable): ', len(hpo_filter))

2026-07-04 12:02:28.563 | DEBUG    | lib.hpo_tree.hpo_loader:get_most_recent_hpo_file:36 - Loading existing *.obo file: data\hp_2026-02-16.obo
2026-07-04 12:02:29.636 | DEBUG    | lib.hpo_tree.hpo_term:load_ontology:285 - Creating HPO tree structure...
2026-07-04 12:02:29.860 | DEBUG    | lib.hpo_tree.hpo_term:load_ontology:298 - HPO tree created!


Annotated HPO-terms: 72
Keep HPO-terms: 96
Amount of HPOs to be used (if trainable):  109


In [149]:
output_files = glob.glob(os.path.join(output_dir, '**', '**', 'result.json'))
result = []
for output_file in tqdm.tqdm(output_files):
    experiment_folder = pathlib.Path(output_file).parts[-2]
    experiment_name = experiment_folder[len('version_'):] if 'version_' in output_file else experiment_folder
    try:
        load = json.load(open(output_file))
        metrics = {node['id']: pd.DataFrame(node['metrics']) for node in load['nodes'] if node['metrics']}
        for node_id, metrics_df in metrics.items():
            mean_df = pd.DataFrame(metrics_df.mean()).transpose()
            std_df = pd.DataFrame(metrics_df.std()).transpose()
            columns = [c.replace('val_', '').replace('class_', '') for c in mean_df.columns]
            mean_df.columns = columns
            std_df.columns = columns
            tmp = {'HPO': node_id, 'experiment': experiment_name}
            db, dim, fo, md, t, sl, s = experiment_folder.split('_')
            tmp['dimensions'] = int(dim.split('=')[1])
            tmp['face outline'] = fo.split('=')[1] == 'True'
            tmp['metadata'] = ', '.join(md.split('=')[1].replace('[', '').replace(']', '').split('+')) if len(
                md) > 2 else 'N/A'
            tmp['threshold'] = float(t.split('=')[1])
            tmp['softlabel'] = float(sl.split('=')[1])
            # tmp = mean_df.copy()  # to keep index/columns
            for col in mean_df.columns[:-1]:
                tmp[f'{col}_mean'] = mean_df[col].item()
                tmp[f'{col}_std'] = std_df[col].item()
            tmp['support'] = int(metrics_df['support'].astype(int).mean())
            result.append(tmp)
    except json.decoder.JSONDecodeError:
        print(f'Skip {experiment_name} => Still in progress.')

df_pointnet = pd.DataFrame(result)
df_filtered = df_pointnet[df_pointnet['HPO'].isin(hpo_filter)]

df_filtered.groupby('experiment').size()

100%|██████████| 72/72 [00:11<00:00,  6.53it/s]


experiment
db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42                       107
db=4_d=2_f=False_m=[]_t=0.01_l=0.05_s=42                       107
db=4_d=2_f=False_m=[]_t=0.01_l=0.10_s=42                       107
db=4_d=2_f=False_m=[]_t=0.05_l=0.00_s=42                       107
db=4_d=2_f=False_m=[]_t=0.05_l=0.05_s=42                       107
                                                              ... 
db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.05_l=0.05_s=42    107
db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.05_l=0.10_s=42    107
db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.10_l=0.00_s=42    107
db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.10_l=0.05_s=42    107
db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.10_l=0.10_s=42    107
Length: 71, dtype: int64

In [150]:
# Build the full expected HPO grid for every experiment configuration
group_cols = ['experiment', 'dimensions', 'face outline', 'metadata', 'threshold', 'softlabel']

existing_configs = df_filtered[group_cols].drop_duplicates()

full_index = existing_configs.assign(key=1).merge(
    pd.DataFrame({'HPO': sorted(df_filtered['HPO'].unique().tolist()), 'key': 1}),
    on='key'
).drop(columns='key')

# Merge observed rows onto the complete grid
df_result_complete = full_index.merge(
    df_filtered,
    on=group_cols + ['HPO'],
    how='left'
)

# Fill missing metrics with -1
non_fill_cols = group_cols + ['HPO']
metric_fill_cols = [c for c in df_result_complete.columns if c not in non_fill_cols]

for col in metric_fill_cols:
    df_result_complete[col] = df_result_complete[col].fillna(0)

# Optional: keep rows ordered nicely
df_result_complete = df_result_complete.sort_values(group_cols + ['HPO']).reset_index(drop=True)

# counts = df_filtered.groupby('experiment').size()
# keep_experiments = counts[counts == 107].index
# df_result_complete = df_filtered[df_filtered['experiment'].isin(keep_experiments)]
df_result_complete

,experiment,dimensions,face outline,metadata,threshold,softlabel,HPO,accuracy_mean,accuracy_std,auroc_mean,...,stat_scores/tp_std,stat_scores/fp_mean,stat_scores/fp_std,stat_scores/tn_mean,stat_scores/tn_std,stat_scores/fn_mean,stat_scores/fn_std,stat_scores/sup_mean,stat_scores/sup_std,support
0,db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42,2,False,,0.01,0.0,HP:0000153,0.789616,0.029293,0.789664,...,5.805170,38.2,6.534524,179.6,7.092249,53.6,10.310189,217.8,7.854935,2178.0
1,db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42,2,False,,0.01,0.0,HP:0000154,0.663353,0.040576,0.661103,...,10.677078,35.0,8.803408,56.8,11.322544,26.8,6.300794,91.8,7.596052,918.0
2,db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42,2,False,,0.01,0.0,HP:0000159,0.794792,0.017464,0.794867,...,8.443933,34.2,4.969909,166.2,6.220932,48.0,7.382412,200.4,6.228965,2004.0
3,db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42,2,False,,0.01,0.0,HP:0000160,0.643670,0.101795,0.651926,...,2.949576,17.4,8.905055,13.0,7.348469,4.2,3.346640,30.4,2.880972,304.0
4,db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42,2,False,,0.01,0.0,HP:0000163,0.792157,0.014626,0.792032,...,9.289779,40.2,4.086563,177.4,4.560702,50.2,3.033150,217.6,6.426508,2176.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7592,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.1...,3,True,"age, gender, ethnicity",0.10,0.1,HP:0100538,0.819865,0.054775,0.819828,...,2.863564,4.6,1.816590,24.4,1.816590,5.8,1.303840,29.0,1.870829,290.0
7593,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.1...,3,True,"age, gender, ethnicity",0.10,0.1,HP:0100840,0.864357,0.026541,0.864555,...,5.099020,7.8,4.816638,63.8,4.604346,11.6,4.615192,71.6,2.073644,716.0
7594,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.1...,3,True,"age, gender, ethnicity",0.10,0.1,HP:0100886,0.852240,0.039342,0.852330,...,8.074652,10.4,4.669047,89.4,4.219005,19.0,5.700877,99.8,3.701351,998.0
7595,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.1...,3,True,"age, gender, ethnicity",0.10,0.1,HP:0200006,0.852074,0.023956,0.851443,...,7.905694,11.6,3.974921,89.8,4.438468,18.4,2.607681,101.4,6.148170,1014.0


In [151]:
df_hpo_frequencies = pd.DataFrame(df_result_complete.groupby('HPO')['support'].mean())
df_hpo_frequencies['HPO'] = [all_hpos[h].name for h in df_hpo_frequencies.index.to_list()]
df_hpo_frequencies['support'] = df_hpo_frequencies['support'].astype(int)
df_hpo_frequencies['support'] = df_hpo_frequencies['support'] // 2
df_hpo_frequencies.rename(columns={'support': 'Samples'}, inplace=True)
df_hpo_frequencies = df_hpo_frequencies[['HPO', 'Samples']]
kappa_df = pd.read_csv('fleiss_kappa_mean_per_hpo.csv')
kappa_df.rename(columns={'hpo': 'HPO'}, inplace=True)
df_hpo_frequencies = df_hpo_frequencies.merge(kappa_df, left_index=True, right_on='HPO', how='left')
df_hpo_frequencies['IRA'] = (df_hpo_frequencies['fleiss_kappa_mean'].map('{:.2f}'.format) + ' ± ' + df_hpo_frequencies['fleiss_kappa_std'].map('{:.2f}'.format) + ' [' + df_hpo_frequencies['fleiss_kappa_ci_low'].map('{:.2f}'.format) + ', ' + df_hpo_frequencies['fleiss_kappa_ci_high'].map('{:.2f}'.format) + ']')
df_hpo_frequencies['IRA'] = df_hpo_frequencies['IRA'].replace(to_replace=r'nan ± nan \[nan, nan\]', value='N/A', regex=True)
df_hpo_frequencies['IRA'] = df_hpo_frequencies['IRA'].replace(to_replace=r' \[nan, nan\]', value='', regex=True)
df_hpo_frequencies = df_hpo_frequencies[['HPO_x', 'HPO', 'IRA', 'Samples']]
df_hpo_frequencies.rename(columns={'HPO_x': 'HPO Name', 'HPO': 'HPO ID'}, inplace=True)
df_hpo_frequencies.to_latex(os.path.join(publication_dir, 'frequency_appendix.tex'), longtable=True, index=False,
                            caption='The frequencies of all annotated HPO terms and their corresponding Inter-Rater-Agreement (IRA) using the mean Fleiss\' Kappa score for each HPO term over all samples.', label='tab:hpo_frequencies_appendix',
                            column_format='rccr')
df_hpo_frequencies

,HPO Name,HPO ID,IRA,Samples
NaN,Abnormality of the mouth,HP:0000153,N/A,1089
22.0,Wide mouth,HP:0000154,"0.80 ± 0.61 [-0.45, 2.05]",459
NaN,Abnormal lip morphology,HP:0000159,N/A,1002
23.0,Narrow mouth,HP:0000160,"0.80 ± 0.61 [-0.45, 2.05]",152
NaN,Abnormal oral cavity morphology,HP:0000163,N/A,1088
...,...,...,...,...
NaN,Abnormality of the supraorbital ridges,HP:0100538,N/A,145
NaN,Aplasia/Hypoplasia of the eyebrow,HP:0100840,N/A,358
NaN,Abnormality of globe location,HP:0100886,N/A,499
NaN,Slanting of the palpebral fissure,HP:0200006,N/A,507


In [152]:
def create_sub_table(dataframe_df: pd.DataFrame, top_n: int = 5, bottom_n: int = 5):
    n_rows = len(dataframe_df)
    if n_rows <= top_n + bottom_n:
        condensed_df = dataframe_df.copy()
    else:
        top = dataframe_df.head(top_n)
        bottom = dataframe_df.tail(bottom_n)

        # Row of "..." between top and bottom
        dots = pd.DataFrame(
            {col: '...' for col in dataframe_df.columns},
            index=['...']
        )

        condensed_df = pd.concat([top, dots, bottom])
    return condensed_df

In [153]:
df_result = df_result_complete.groupby('experiment', as_index=False)[[metric_mean, metric_std]].mean().sort_values(
    by=[metric_mean], ascending=False)

ablation_study_results = {'D': [], 'FO': [], 'Metadata': [], 'T': [], 'S': [], 'mean AUROC': [], 'AUROC': []}
for _, row in df_result.iterrows():
    db, dim, faceoutline, metadata, threshold, softlabel, seed = (n.split('=')[1] for n in row['experiment'].split('_'))
    ablation_study_results['D'].append(int(dim))
    ablation_study_results['FO'].append(faceoutline)
    metadata_data = ', '.join(metadata.replace('[', '').replace(']', '').split('+'))
    ablation_study_results['Metadata'].append(metadata_data if len(metadata_data) > 0 else 'N/A')
    ablation_study_results['T'].append(float(threshold))
    ablation_study_results['S'].append(float(softlabel))
    auroc_mean = row['auroc_mean']
    auroc_std = row['auroc_std']
    auroc_str = f'{auroc_mean:.03f} ± {auroc_std:.03f}'
    ablation_study_results['mean AUROC'].append(auroc_str)
    ablation_study_results['AUROC'].append(float(auroc_mean))

ablation_study_result_df = pd.DataFrame(ablation_study_results).sort_values(by=['AUROC'], ascending=False).reset_index(
    drop=True).drop(
    columns=['AUROC'])
latex = ablation_study_result_df.to_latex(float_format='%.2f', longtable=True, index=False,
                                          caption='The results of all experiments from the ablation study. D: Dimension, FO: Face Outline, T: Feature Importance Threshold, S: Softlabel',
                                          label='tab:ablation_study_appendix',
                                          column_format='c' * (len(ablation_study_result_df.columns) + 1))

latex = latex.replace(' ± ', '$\\pm$')

with open(os.path.join(publication_dir, 'ablation_study_appendix.tex'), 'w') as f:
    f.write(latex)

condensed_df = create_sub_table(ablation_study_result_df, top_n=10, bottom_n=5)

latex = condensed_df.to_latex(
    float_format='%.2f',
    longtable=False,
    index=False,
    caption='The top-10 and bottom-5 results of the ablation study with 72 experiments. D=Dimensions, FO=Face Outline, T=Feature Importance Threshold, S=Soft Label.',
    label='tab:ablation_top10',
    column_format='c' * len(condensed_df.columns)
)

latex = latex.replace(' ± ', '$\\pm$')

with open(os.path.join(publication_dir, 'ablation_study.tex'), 'w') as f:
    f.write(latex)

df_result

# fig, axes = plt.subplots(1, 1, figsize=(4, 20))
# n_base = 20
# color_map = 'tab20'
# cmap = plt.colormaps[color_map]
# base_colors = cmap(np.linspace(0, 1, n_base))
# recycled_cmap = ListedColormap(base_colors)
# colors = recycled_cmap(np.arange(len(df_result)) % n_base)
#
# # Ersetze deinen plotting code komplett:
# y_count = np.arange(len(df_result))
# bars = axes.barh(y_count, df_result[metric_mean],
#                  xerr=df_result[metric_std],
#                  color=colors, alpha=0.8)
# axes.set_xlabel(f'Mean AUROC')
# axes.set_ylabel('Experiment')
# padding = 0.5  # Padding oben/unten
# axes.set_ylim(-padding, len(df_result) + padding - 1)
# axes.set_yticks(y_count, df_result['experiment'])
# axes.set_xlim(0, 1)
# axes.invert_yaxis()
#
# bar_height = 0.65  # Höhe pro Bar
# for bar, mean, std in zip(bars, df_result[metric_mean], df_result[metric_std]):
#     if mean > 0:
#         axes.text(mean // 2 + 0.01, bar.get_y() + bar_height / 2,
#                   f'{mean:.2f} ± {std:.2f}', va='center', ha='left', fontsize=10)
#
# plt.show()
# plt.close()

,experiment,auroc_mean,auroc_std
63,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,0.749444,0.044016
67,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,0.747460,0.043510
64,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,0.747089,0.042927
66,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,0.740694,0.043387
45,db=4_d=3_f=False_m=[age+gender+ethnicity]_t=0....,0.740229,0.044544
...,...,...,...
8,db=4_d=2_f=False_m=[]_t=0.10_l=0.10_s=42,0.653278,0.054501
7,db=4_d=2_f=False_m=[]_t=0.10_l=0.05_s=42,0.651169,0.054874
24,db=4_d=2_f=True_m=[]_t=0.10_l=0.00_s=42,0.647687,0.051478
26,db=4_d=2_f=True_m=[]_t=0.10_l=0.10_s=42,0.645549,0.050130


# Ablation Study with investigated Statistical Significances

In [154]:
import os
import math
import numpy as np
import pandas as pd
from scipy import stats

# Explicitly define which columns hold the metric
metric_mean_col = 'auroc_mean'
metric_std_col = 'auroc_std'

group_cols = ['experiment']

# Aggregate per experiment across HPO terms (or folds)
df_result = (
    df_result_complete
    .groupby(group_cols)[metric_mean_col]
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .rename(columns={'mean': 'auroc_mean', 'std': 'auroc_std', 'count': 'n'})
)

# 95% confidence interval using t-distribution
df_result['auroc_sem'] = df_result['auroc_std'] / np.sqrt(df_result['n'])
df_result['t_crit'] = df_result['n'].apply(
    lambda n: stats.t.ppf(0.975, df=n - 1) if n > 1 else np.nan
)
df_result['auroc_ci95_low'] = df_result['auroc_mean'] - df_result['t_crit'] * df_result['auroc_sem']
df_result['auroc_ci95_high'] = df_result['auroc_mean'] + df_result['t_crit'] * df_result['auroc_sem']

# Keep a sorted version for ranking
df_result = df_result.sort_values(by='auroc_mean', ascending=False).reset_index(drop=True)

# Pairwise comparison against the best experiment
best_experiment = df_result.loc[0, 'experiment']

# All per-term AUROC means for the best experiment
best_values = df_result_complete.loc[
    df_result_complete['experiment'] == best_experiment, metric_mean_col
].dropna()

p_values = []
effect_sizes = []

for _, row in df_result.iterrows():
    current_experiment = row['experiment']
    current_values = df_result_complete.loc[
        df_result_complete['experiment'] == current_experiment, metric_mean_col
    ].dropna()

    if current_experiment == best_experiment:
        p_values.append(np.nan)
        effect_sizes.append(np.nan)
        continue

    # Welch's t-test
    if len(best_values) > 1 and len(current_values) > 1:
        _, p_val = stats.ttest_ind(best_values, current_values, equal_var=False)
    else:
        p_val = np.nan
    p_values.append(p_val)

    # Cohen's d (pooled SD)
    n1, n2 = len(best_values), len(current_values)
    if n1 > 1 and n2 > 1:
        s1, s2 = best_values.std(ddof=1), current_values.std(ddof=1)
        pooled_sd = math.sqrt(((n1 - 1) * s1 ** 2 + (n2 - 1) * s2 ** 2) / (n1 + n2 - 2))
        d = (best_values.mean() - current_values.mean()) / pooled_sd if pooled_sd > 0 else np.nan
    else:
        d = np.nan
    effect_sizes.append(d)

df_result['p'] = p_values
df_result['cohens_d'] = effect_sizes

# Optional multiple-testing correction
valid_mask = df_result['p'].notna()
pvals = df_result.loc[valid_mask, 'p'].values

if len(pvals) > 0:
    from statsmodels.stats.multitest import multipletests

    _, pvals_adj, _, _ = multipletests(pvals, method='fdr_bh')
    df_result.loc[valid_mask, 'p_fdr'] = pvals_adj
else:
    df_result['p_fdr'] = np.nan

ablation_study_results = {
    'D': [],
    'FO': [],
    'Metadata': [],
    'T': [],
    'S': [],
    'mean AUROC': [],
    '95\\% CI': [],
    'p': [],
    'FDR p': [],
    'd': [],
    'AUROC': []
}

for _, row in df_result.iterrows():
    # experiment string pattern: db=4_d=2_f=False_m=[]_t=0.01_l=0.00_s=42
    db, dim, faceoutline, metadata, threshold, softlabel, seed = (
        n.split('=')[1] for n in row['experiment'].split('_')
    )

    ablation_study_results['D'].append(int(dim))
    ablation_study_results['FO'].append(faceoutline)

    metadata_data = ', '.join(metadata.replace('[', '').replace(']', '').split('+'))
    ablation_study_results['Metadata'].append(metadata_data if len(metadata_data) > 0 else 'N/A')

    ablation_study_results['T'].append(float(threshold))
    ablation_study_results['S'].append(float(softlabel))

    auroc_mean = row['auroc_mean']
    auroc_std = row['auroc_std']
    ci_low = row['auroc_ci95_low']
    ci_high = row['auroc_ci95_high']

    ablation_study_results['mean AUROC'].append(f'{auroc_mean:.3f} $\\pm$ {auroc_std:.3f}')
    ablation_study_results['95\\% CI'].append(f'[{ci_low:.3f}, {ci_high:.3f}]')

    p_val = row['p']
    p_val_fdr = row.get('p_fdr', np.nan)
    d_val = row['cohens_d']

    ablation_study_results['p'].append('—' if pd.isna(p_val) else f'{p_val:.3g}')
    ablation_study_results['FDR p'].append('—' if pd.isna(p_val_fdr) else f'{p_val_fdr:.3g}')
    ablation_study_results['d'].append('—' if pd.isna(d_val) else f'{d_val:.2f}')
    ablation_study_results['AUROC'].append(float(auroc_mean))

ablation_study_result_df = (
    pd.DataFrame(ablation_study_results)
    .sort_values(by=['AUROC'], ascending=False)
    .reset_index(drop=True)
)

full_table_df = ablation_study_result_df.drop(columns=['AUROC'])

latex = full_table_df.to_latex(
    float_format='%.2f',
    longtable=True,
    escape=False,
    index=False,
    caption=(
        'The results of all experiments from the ablation study. '
        'D: Dimension, FO: Face Outline, T: Feature Importance Threshold, '
        'S: Soft Label. Confidence intervals are 95\\% t-based intervals. '
        'p-values compare each experiment against the best-performing experiment using Welch\\textquotesingle s t-test; '
        'FDR p-values are Benjamini--Hochberg corrected; Cohen\\textquotesingle s d highlights how large and '
        'practically meaningful the difference is between two configurations.'
    ),
    label='tab:ablation_study_appendix',
    column_format='c' * len(full_table_df.columns)
)

with open(os.path.join(publication_dir, 'ablation_study_appendix.tex'), 'w') as f:
    f.write(latex)

condensed_df = create_sub_table(full_table_df, top_n=10, bottom_n=5)

latex = condensed_df.to_latex(
    float_format='%.2f',
    escape=False,
    longtable=False,
    index=False,
    caption=(
        'The top-10 and bottom-5 results of the ablation study with 72 experiments. '
        'D: Dimensions, FO: Face Outline, T: Feature Importance Threshold, S: Soft Label. '
        'Confidence intervals are 95\\% t-based intervals; statistical comparisons are against the best experiment; '
        'Cohen\\textquotesingle s d highlights how large and practically meaningful the difference is between two configurations.'
    ),
    label='tab:ablation_top10',
    column_format='c' * len(condensed_df.columns)
)

with open(os.path.join(publication_dir, 'ablation_study.tex'), 'w') as f:
    f.write(latex)

full_table_df

,D,FO,Metadata,T,S,mean AUROC,95\% CI,p,FDR p,d
0,3,True,"age, gender, ethnicity",0.01,0.00,0.749 $\pm$ 0.108,"[0.729, 0.770]",—,—,—
1,3,True,"age, gender, ethnicity",0.05,0.10,0.747 $\pm$ 0.108,"[0.727, 0.768]",0.893,0.893,0.02
2,3,True,"age, gender, ethnicity",0.01,0.10,0.747 $\pm$ 0.111,"[0.726, 0.768]",0.875,0.887,0.02
3,3,True,"age, gender, ethnicity",0.05,0.05,0.741 $\pm$ 0.110,"[0.720, 0.762]",0.558,0.574,0.08
4,3,False,"age, gender, ethnicity",0.01,0.00,0.740 $\pm$ 0.107,"[0.720, 0.761]",0.532,0.555,0.09
...,...,...,...,...,...,...,...,...,...,...
66,2,False,N/A,0.10,0.10,0.653 $\pm$ 0.105,"[0.633, 0.673]",2.78e-10,3.9e-09,0.91
67,2,False,N/A,0.10,0.05,0.651 $\pm$ 0.103,"[0.631, 0.671]",9.56e-11,1.67e-09,0.93
68,2,True,N/A,0.10,0.00,0.648 $\pm$ 0.108,"[0.627, 0.668]",6.01e-11,1.4e-09,0.94
69,2,True,N/A,0.10,0.10,0.646 $\pm$ 0.109,"[0.625, 0.666]",3e-11,1.05e-09,0.96


In [155]:
best_model_config = \
    df_result_complete.groupby('experiment')[[metric_mean, metric_std]].mean().sort_values(by=[metric_mean],
                                                                                           ascending=False).iloc[
        0].name
best_model = df_result_complete[df_result_complete['experiment'] == best_model_config]
additional_params = {'inputpoints': [], 'parameters': []}
for hpo_id in best_model['HPO'].unique():
    hpo_df = best_model[best_model['HPO'] == hpo_id]
    exp_base_dir = os.path.join(output_dir, hpo_id.replace(':', '_'), hpo_df['experiment'].values[0])
    pmi = np.load(os.path.join(exp_base_dir, 'point_mask_input.npy'))
    num_in_points = pmi.sum()
    conf = json.load(open(os.path.join(exp_base_dir, 'config.json'), 'r'))
    model_spec = create_model(pmi, conf['dimensions'], conf['meta_data'])
    param_count = sum(p.numel() for p in model_spec.parameters())
    additional_params['inputpoints'].append(num_in_points)
    additional_params['parameters'].append(param_count)

best_model = best_model.assign(**additional_params)
best_model

,experiment,dimensions,face outline,metadata,threshold,softlabel,HPO,accuracy_mean,accuracy_std,auroc_mean,...,stat_scores/fp_std,stat_scores/tn_mean,stat_scores/tn_std,stat_scores/fn_mean,stat_scores/fn_std,stat_scores/sup_mean,stat_scores/sup_std,support,inputpoints,parameters
6741,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000153,0.878014,0.019714,0.877754,...,7.733046,201.2,8.438009,36.6,7.092249,217.8,7.854935,2178.0,136,2754250
6742,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000154,0.715096,0.051628,0.715666,...,6.685806,54.0,2.828427,14.4,3.577709,91.8,7.596052,918.0,82,562026
6743,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000159,0.883121,0.012987,0.883130,...,5.899152,182.8,6.978539,29.2,1.788854,200.4,6.228965,2004.0,106,2754250
6744,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000160,0.745653,0.072951,0.744755,...,2.880972,20.8,3.701351,5.8,3.114482,30.4,2.880972,304.0,82,562026
6745,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000163,0.880831,0.016762,0.880758,...,3.847077,199.0,3.391165,33.2,6.610598,217.6,6.426508,2176.0,112,2754250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6843,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100538,0.799407,0.057856,0.799186,...,2.000000,25.0,2.121320,7.6,1.949359,29.0,1.870829,290.0,90,562026
6844,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100840,0.881073,0.030987,0.881248,...,4.301163,64.6,3.911521,10.0,2.345208,71.6,2.073644,716.0,292,2754250
6845,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100886,0.873534,0.031620,0.873097,...,2.302173,89.2,2.683282,14.6,5.176872,99.8,3.701351,998.0,68,562026
6846,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0200006,0.875811,0.019595,0.875986,...,1.303840,91.6,1.140175,15.4,4.037326,101.4,6.148170,1014.0,57,562026


In [156]:
# Compute Prevalence
result = {}
best_model_result_file = os.path.join(output_dir, 'HP_0000271', best_model_config, 'result.json')
experiment_folder = pathlib.Path(best_model_result_file).parts[-2]
experiment_name = experiment_folder[len('version_'):] if 'version_' in best_model_result_file else experiment_folder
load = json.load(open(best_model_result_file))
metrics = {node['id']: pd.DataFrame(node['metrics']) for node in load['nodes'] if node['metrics']}
for id, metrics_df in metrics.items():
    if id in hpo_filter:
        stat_scores_cols = [n for n in metrics_df.columns if 'stat_scores' in n and 'sup' not in n]
        tp, fp, tn, fn = (metrics_df[col].sum() for col in stat_scores_cols)
        prev = (tp + fn) / (tp + fn + tn + fp)  # prevalence
        prev_det = (tp + fp) / (tp + fn + tn + fp)  # detection prevalence
        result.update({id: {
            # 'Prev': prev.item(),
            'Det. P': prev_det.item()
        }})
prev_all_df = pd.DataFrame(result).transpose().sort_values(by=['Det. P'])
# prev_all_df.to_latex(os.path.join(publication_dir, 'train_val_prev.tex'), float_format='%.03f', longtable=True)
prev_all_df

,Det. P
HP:0000568,0.287879
HP:0003189,0.417391
HP:0000581,0.427632
HP:0100538,0.437931
HP:0000499,0.443709
...,...
HP:0000280,0.648214
HP:0000179,0.649321
HP:0000574,0.658501
HP:0000455,0.680162


In [157]:
kappa_df = pd.read_csv('fleiss_kappa_mean_per_hpo.csv')
kappa_df.rename(columns={'hpo': 'HPO'}, inplace=True)
df_kappa = best_model.merge(kappa_df, on='HPO', how='left')
df_kappa

,experiment,dimensions,face outline,metadata,threshold,softlabel,HPO,accuracy_mean,accuracy_std,auroc_mean,...,stat_scores/sup_mean,stat_scores/sup_std,support,inputpoints,parameters,fleiss_kappa_mean,fleiss_kappa_std,fleiss_kappa_ci_low,fleiss_kappa_ci_high,num_images
0,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000153,0.878014,0.019714,0.877754,...,217.8,7.854935,2178.0,136,2754250,NaN,NaN,NaN,NaN,NaN
1,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000154,0.715096,0.051628,0.715666,...,91.8,7.596052,918.0,82,562026,0.8,0.610257,-0.448116,2.048116,30.0
2,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000159,0.883121,0.012987,0.883130,...,200.4,6.228965,2004.0,106,2754250,NaN,NaN,NaN,NaN,NaN
3,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000160,0.745653,0.072951,0.744755,...,30.4,2.880972,304.0,82,562026,0.8,0.610257,-0.448116,2.048116,30.0
4,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0000163,0.880831,0.016762,0.880758,...,217.6,6.426508,2176.0,112,2754250,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100538,0.799407,0.057856,0.799186,...,29.0,1.870829,290.0,90,562026,NaN,NaN,NaN,NaN,NaN
103,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100840,0.881073,0.030987,0.881248,...,71.6,2.073644,716.0,292,2754250,NaN,NaN,NaN,NaN,NaN
104,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0100886,0.873534,0.031620,0.873097,...,99.8,3.701351,998.0,68,562026,NaN,NaN,NaN,NaN,NaN
105,db=4_d=3_f=True_m=[age+gender+ethnicity]_t=0.0...,3,True,"age, gender, ethnicity",0.01,0.0,HP:0200006,0.875811,0.019595,0.875986,...,101.4,6.148170,1014.0,57,562026,NaN,NaN,NaN,NaN,NaN


In [158]:
focus = ['auroc_mean', 'inputpoints', 'fleiss_kappa_mean', 'support', 'parameters']

df_best = df_kappa.copy()

# Identify top/bottom 5
top5_idx = df_best.nlargest(5, 'auroc_mean').index
bottom5_idx = df_best.nsmallest(5, 'auroc_mean').index
others_idx = ~df_best.index.isin(top5_idx.union(bottom5_idx))

top5_ids = df_best.loc[top5_idx, 'HPO'].values.tolist()
top5_ids = [all_hpos[hpo_id] for hpo_id in top5_ids]
print(top5_ids)
bottom5_ids = df_best.loc[bottom5_idx, 'HPO'].values.tolist()
print(bottom5_ids)
bottom5_ids = [all_hpos[hpo_id] for hpo_id in bottom5_ids]
print(bottom5_ids)

# Summary (unchanged)
summary = df_best[focus].describe().round(3)
corr = df_best[focus].corr().round(3)

print('Dataset: 107 HPO experiments from ML tuning (age/gender/ethnicity features).')
print(summary)
print('\nCorrelations:\n', corr['auroc_mean'])

# Visualize with highlighting + trendlines
handles = []
labels = []
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
colors = {'top': 'green', 'bottom': 'orange', 'other': 'lightblue'}
for i, col in enumerate(focus[1:], 0):
    r = corr['auroc_mean'][col]
    ax = axes[i // 2, i % 2]
    # Scatter with colors
    sc_others = ax.scatter(df_best.loc[others_idx, col], df_best.loc[others_idx, 'auroc_mean'],
               c=colors['other'], alpha=0.6, s=40, label='Others', edgecolors='gray')
    sc_top = ax.scatter(df_best.loc[top5_idx, col], df_best.loc[top5_idx, 'auroc_mean'],
               c=colors['top'], alpha=0.9, s=100, marker='^', label='Top-5', edgecolors='darkgreen', linewidth=1.5)
    sc_bottom = ax.scatter(df_best.loc[bottom5_idx, col], df_best.loc[bottom5_idx, 'auroc_mean'],
               c=colors['bottom'], alpha=0.9, s=100, marker='v', label='Bottom-5', edgecolors='darkorange',
               linewidth=1.5)

    if i == 0:
        handles = [sc_others, sc_top, sc_bottom]
        labels = ['Others', 'Top-5', 'Bottom-5']

    # Trendline if strong correlation
    if abs(r) > 0.3:
        r = corr['auroc_mean'][col]
        slope, intercept, r_val, p_val, _ = linregress(df_best[col], df_best['auroc_mean'])
        x_fit = np.linspace(df_best[col].min(), df_best[col].max(), 100)
        y_fit = slope * x_fit + intercept
        ax.plot(x_fit, y_fit, 'red', linewidth=2.5, label=f'r={r:.3f}')

    ax.set_xlabel(col.replace('support', '# of samples').replace('inputpoints', '# of points').replace('fleiss_kappa_mean', 'IRA'), fontsize=14)
    ax.set_ylabel('mean AUROC', fontsize=14)
    ax.set_xlim(0, df_best[col].max() * 1.05)
    ax.set_ylim(0.5, 1)
    ax.grid(True, alpha=0.3)
    # ax.legend()
    # else:
    #     fig.delaxes(ax)

revised_labels = ['mean AUROC', '# of points', 'IRA', '# of samples', '# of parameters']

# Create a mask for the upper triangle (excluding the diagonal)
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    center=0,
    ax=axes[1, 1],
    vmin=-1,
    vmax=1,
    xticklabels=revised_labels,
    yticklabels=revised_labels,
    mask=mask,          # <-- only show lower triangle + diagonal
    square=True
)

axes[1, 1].set_title('Correlation Matrix', fontsize=14)
axes[1, 1].grid(visible=False)

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.521, 0.585))

plt.suptitle('Key Metrics: Top-5 (Green ▲), Bottom-5 (Orange ▼), Trendlines (|r|>0.3)', fontsize=14)
plt.tight_layout()
plt.show()
plt.savefig(os.path.join(publication_dir, 'pearson_correlation.pdf'), format='pdf', dpi=300)
plt.close()

[HP:0000492 (Abnormal eyelid morphology), HP:0031816 (Abnormal oral morphology), HP:0012372 (Abnormal eye morphology), HP:0010938 (Abnormal external nose morphology), HP:0000159 (Abnormal lip morphology)]
['HP:0000525', 'HP:0004428', 'HP:0000568', 'HP:0010805', 'HP:0000581']
[HP:0000525 (Abnormality iris morphology), HP:0004428 (Elfin facies), HP:0000568 (Microphthalmia), HP:0010805 (Upturned corners of mouth), HP:0000581 (Blepharophimosis)]
Dataset: 107 HPO experiments from ML tuning (age/gender/ethnicity features).
       auroc_mean  inputpoints  fleiss_kappa_mean   support   parameters
count     107.000      107.000             62.000   107.000      107.000
mean        0.749      117.514              0.630   789.215  1504477.439
std         0.108       77.038              0.247   564.780  1090395.222
min         0.555       54.000              0.029    66.000   562026.000
25%         0.661       70.000              0.432   383.000   562026.000
50%         0.719       88.000         

In [159]:
bm = df_kappa.sort_values(by=[metric_mean], ascending=False)
df_best_model_hpos = pd.DataFrame({
    #"HPO Description": [hpo.find_successor(id).name for id in bm['HPO'].tolist()],
    "HPO": bm["HPO"],
    "AUROC": bm['auroc_mean'].map('{:.2f}'.format) + ' ± ' + bm['auroc_std'].map('{:.2f}'.format),
    "F1-Score": bm['f1_score_mean'].map('{:.2f}'.format) + ' ± ' + bm['f1_score_std'].map('{:.2f}'.format),
    "Precision": bm['precision_mean'].map('{:.2f}'.format) + ' ± ' + bm['precision_std'].map('{:.2f}'.format),
    "Recall": bm['recall_mean'].map('{:.2f}'.format) + ' ± ' + bm['recall_std'].map('{:.2f}'.format),
    "N": bm["support"].astype(int),
}).set_index('HPO')
df_best_model_hpos = df_best_model_hpos[['AUROC', 'F1-Score', 'Precision', 'Recall', 'N']]
df_best_model_hpos.index.name = None
df_best_model_hpos = pd.merge(df_best_model_hpos, prev_all_df, left_index=True, right_index=True, how='inner')
df_best_model_hpos.index = [all_hpos[i].name for i in df_best_model_hpos.index.tolist()]
latex = df_best_model_hpos.to_latex(float_format='%.2f', longtable=True, label='tab:best_model_hpos_appendix',
                                    caption='The results of all HPO models with the best configuration. The HPO terms in bold letters are leafs of the human phenotype ontology. N: Samples, Prev: Prevalence, Det. P: Detection Prevalence',
                                    column_format='r' + 'c' * len(df_best_model_hpos.columns))

for lh in leaf_hpos:
    latex = latex.replace(f'{lh.name} ', f'\\textbf{{{lh.name}}} ')
latex = latex.replace(' ± ', '$\\pm$')

with open(os.path.join(publication_dir, 'best_model_hpos_appendix.tex'), 'w') as f:
    f.write(latex)

leaf_node_names = [n.name for n in leaf_hpos]
keep_indexes = [n for n in df_best_model_hpos.index.tolist() if n in leaf_node_names]
condensed_df = create_sub_table(df_best_model_hpos.loc[keep_indexes], top_n=5, bottom_n=5)
print('Number of leaf HPO terms: ', len(df_best_model_hpos.loc[keep_indexes]))

latex = condensed_df.to_latex(
    float_format='%.2f',
    longtable=False,
    caption='An overview of the HPO models of the best model configuration, with a selection of the models with their top-5 and bottom-5 performing leaf HPO models and their mean scores over the 5-Fold Cross Validation. The samples (S) are 50\\% affected and 50\\% unaffected. The detection prevalence (Det. P) shows the model\'s perception of prevalence.',
    label='tab:best_model_hpos',
    column_format='r' + 'c' * len(condensed_df.columns)
)

for lh in leaf_hpos:
    latex = latex.replace(f'{lh.name} ', f'\\textbf{{{lh.name}}} ')
latex = latex.replace(' ± ', '$\\pm$')

with open(os.path.join(publication_dir, 'best_model_hpos.tex'), 'w') as f:
    f.write(latex)

df_best_model_hpos

Number of leaf HPO terms:  59


,AUROC,F1-Score,Precision,Recall,N,Det. P
Abnormal eyelid morphology,0.89 ± 0.02,0.88 ± 0.02,0.89 ± 0.00,0.88 ± 0.03,1766,0.494904
Abnormal oral morphology,0.88 ± 0.02,0.88 ± 0.02,0.91 ± 0.03,0.85 ± 0.03,2176,0.463695
Abnormal eye morphology,0.88 ± 0.03,0.88 ± 0.03,0.90 ± 0.04,0.87 ± 0.03,1090,0.483486
Abnormal external nose morphology,0.88 ± 0.01,0.88 ± 0.01,0.89 ± 0.01,0.87 ± 0.02,1722,0.486063
Abnormal lip morphology,0.88 ± 0.01,0.88 ± 0.01,0.91 ± 0.03,0.85 ± 0.01,2004,0.471058
...,...,...,...,...,...,...
Blepharophimosis,0.57 ± 0.10,0.46 ± 0.25,0.53 ± 0.03,0.51 ± 0.38,152,0.427632
Upturned corners of mouth,0.56 ± 0.10,0.57 ± 0.25,0.52 ± 0.17,0.65 ± 0.33,244,0.594262
Microphthalmia,0.56 ± 0.12,0.33 ± 0.36,0.34 ± 0.38,0.32 ± 0.34,66,0.287879
Elfin facies,0.56 ± 0.09,0.42 ± 0.39,0.37 ± 0.33,0.51 ± 0.47,216,0.449074


In [160]:
def find_threshold(df, group_mask, threshold):
    """Find smallest support where all points to RIGHT have score >= threshold."""
    group_df = df[group_mask].copy()
    return group_df[group_df[metric_mean] < threshold]['support'].max()

config = {p.split('=')[0]: p.split('=')[-1] for p in best_model_config.split('_')}
dimension = int(config['d'])
# face_outline = bool(config['f'])
meta_data = config['m'][1:-1].split('+')
# threshold = float(config['t'])
# softlabel = float(config['l'])

hpo_models = HumanPhenotypeModel.create_from_hpo(hp_abnorm_face, output_dir, dimensions=dimension, meta_data=meta_data,
                                                 parallel=False, version=best_model_config, max_num_workers=1,
                                                 recursive=True)
parent_hpo = [h.id for h in hpo_models.list_parent_nodes()]
compression_hpo = [h.id for h in hpo_models.list_compression_nodes()]
leaf_hpo = [h.id for h in hpo_models.list_leaf_nodes()]

plt.figure(figsize=(6, 4))
mask_parent = best_model["HPO"].isin(parent_hpo)
plt.scatter(best_model.loc[mask_parent, "support"],
            best_model.loc[mask_parent, metric_mean],
            c='skyblue', alpha=0.7, s=40, edgecolors='white', label=f"Parent ({len(parent_hpo)})")

mask_compression = best_model["HPO"].isin(compression_hpo)
plt.scatter(best_model.loc[mask_compression, "support"],
            best_model.loc[mask_compression, metric_mean],
            c='orange', alpha=0.5, s=40, edgecolors='white', label=f"Compression ({len(compression_hpo)})")

mask_leaf = best_model["HPO"].isin(leaf_hpo)
plt.scatter(best_model.loc[mask_leaf, "support"],
            best_model.loc[mask_leaf, metric_mean],
            c='magenta', alpha=0.9, s=40, edgecolors='white', label=f"Leaf ({len(leaf_hpo)})")

# Calculate for both groups
threshold = 0.75
thresh_compression = find_threshold(best_model, mask_compression, threshold)
thresh_parent = find_threshold(best_model, mask_parent, threshold)
thresh_leaf = find_threshold(best_model, mask_leaf, threshold)

plt.axvline(x=thresh_parent, color='skyblue', linestyle='--', linewidth=2, alpha=0.5,
            label=f'All >={round(threshold * 100, 1)}% (>={int(thresh_parent)} samples)')
plt.axvline(x=thresh_compression, color='orange', linestyle='--', linewidth=2, alpha=0.5,
            label=f'All >={round(threshold * 100, 1)}% (>={int(thresh_compression)} samples)')
plt.axvline(x=thresh_leaf, color='magenta', linestyle='--', linewidth=2, alpha=0.5,
            label=f'All >={round(threshold * 100, 1)}% (>={int(thresh_leaf)} samples)')

plt.ylim(0, 1.1)
plt.xlabel("Support (number of samples)")
plt.ylabel(f"Mean AUROC")
# plt.title(f"Experiment: {best_model_config}\nMean {metric_mean} vs. Support")
plt.grid(True, linestyle="--", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(publication_dir, 'correlation_support_auroc.pdf'), format='pdf', dpi=300)
plt.show()
plt.close()

# Testing on "external" Dataset

In [161]:
reference_face_file = 'data/reference_face.jpg'

reference_face_mesh = extract_face_meshes([reference_face_file])
reference_face_mesh = reference_face_mesh[1].reshape((-1, 3))[:, :dimension]

Face Mesh Extraction: 100%|██████████| 1/1 [00:00<00:00, 58.73it/s]


In [162]:
all_hpos_available = {h.id: h for h in hpo.all_successors(with_self=True)}

def add_all_hpos(row):
    not_predictable = []
    hpos = set(ast.literal_eval(row['hpos']))
    for hpo_term in hpos.copy():
        if hpo_term in all_hpos_available:
            successor = all_hpos_available[hpo_term]
            predecessors = successor.predecessors()
            hpos.update([p.id for p in predecessors])
        else:
            not_predictable.append(hpo_term)
    row['name'] = int(row['name'])
    row['hpos'] = [h for h in list(hpos) if h not in not_predictable]
    if len(row['hpos']) > 0 and 'HP:0000271' not in row['hpos']:
        raise Exception(f'The {hpo.id} must be included in {row['hpos']}')
    return row

def load_annotations(annotators: int, drop_empty_hpos: bool = False):
    annotations = pd.read_csv(f'data/testing/Face2HPO/nih_hpo_labels_converted_1.06.2023_n{annotators}.tsv', sep='\t')
    # annotations['hpos_with_predecessors'] = None
    annotations = annotations.apply(add_all_hpos, axis=1)
    annotations['annotators'] = [annotators] * len(annotations)
    if drop_empty_hpos:
        annotations = annotations[annotations['hpos'].apply(len) > 0]
    return annotations

filter_empty_hpos = True
df = load_annotations(2, filter_empty_hpos)  # pd.concat([load_annotations(n, filter_empty_hpos) for n in range(1, 4, 1)])

test_complete = set(df['name'].unique().tolist())

images = glob.glob('data/testing/**/*.jpg', recursive=True)
images = {int(pathlib.Path(image_path).stem): image_path for image_path in images if
          int(pathlib.Path(image_path).stem) in df['name'].tolist()}
images = {id: path for id, path in images.items() if id in df_gmdb['image_id'].tolist()}
print('Found: ', len(images), ' images')

df = df[df['name'].isin(images.keys())]
test_filtered = set(df['name'].unique().tolist())

disorder_mapping = {
    'Coffin-Siris': 'Coffin-Siris syndrome (CSS)',
    'Cornelia de Lange': 'Cornelia de Lange syndrome (CdLS)',
    'Ogden': 'Ogden syndrome (OGDNS)',
    'FBXW7': 'FBXW7 syndrome (FBXW7S)',
    'Floating-Harbor': 'Floating-Harbor syndrome (FLHS)',
    'Hyperphosphatasia': 'Hyperphosphatasia with mental retardation syndrome (HPMRS)',
    'Mowat-Wilson syndrome': 'Mowat-Wilson syndrome (MOWS)',
    'Nicolaides-Baraitser': 'Nicolaides-Baraitser syndrome (NCBRS)',
    'SBBYS': 'Ohdo syndrome, SBBYS-variant (SBBYSS)',
    'Opitz GBBB': 'Opitz GBBB syndrome (OGBBBS)',
    'KBG': 'KBG syndrome (KBGS)',
    'Noonan': 'Noonan syndrome (NS)',
    'Williams-Beuren': 'Williams-Beuren syndrome (WBS)',
    'Sotos': 'Sotos syndrome (SOTOS)',
    'White-Sutton': 'White-Sutton syndrome (WHSUS)',
    'Austism': 'Susceptibility to Autism (StA)',
    'Seckel': 'Seckel syndrome (SS)'
}

df['disorder_name'] = [[v for k, v in disorder_mapping.items() if k.lower() in d.lower()][0] for d in
                       df['disorder_name'].tolist()]
print('Differences: ', test_complete - test_filtered)
print('Disorders: ', df['disorder_name'].unique().tolist())
print('Patients per Disorders: ', df['disorder_name'].value_counts())
df

Found:  75  images
Differences:  {1249, 2017, 8934, 1255, 8938, 4667, 5437}
Disorders:  ['Susceptibility to Autism (StA)', 'Coffin-Siris syndrome (CSS)', 'Cornelia de Lange syndrome (CdLS)', 'FBXW7 syndrome (FBXW7S)', 'Floating-Harbor syndrome (FLHS)', 'Hyperphosphatasia with mental retardation syndrome (HPMRS)', 'KBG syndrome (KBGS)', 'Mowat-Wilson syndrome (MOWS)', 'Nicolaides-Baraitser syndrome (NCBRS)', 'Noonan syndrome (NS)', 'Ogden syndrome (OGDNS)', 'Ohdo syndrome, SBBYS-variant (SBBYSS)', 'Opitz GBBB syndrome (OGBBBS)', 'Sotos syndrome (SOTOS)', 'Seckel syndrome (SS)', 'White-Sutton syndrome (WHSUS)', 'Williams-Beuren syndrome (WBS)']
Patients per Disorders:  disorder_name
Coffin-Siris syndrome (CSS)                                   10
Cornelia de Lange syndrome (CdLS)                              5
FBXW7 syndrome (FBXW7S)                                        5
Floating-Harbor syndrome (FLHS)                                5
Hyperphosphatasia with mental retardation syndrome

,name,disorder_name,hpos,annotators
3,7258,Susceptibility to Autism (StA),"[HP:0000271, HP:0000478, HP:0000316, HP:010088...",2
5,7263,Susceptibility to Autism (StA),"[HP:0000153, HP:0000271, HP:0000163, HP:000015...",2
6,8862,Coffin-Siris syndrome (CSS),"[HP:0000271, HP:0000163, HP:0000219, HP:001281...",2
7,3515,Coffin-Siris syndrome (CSS),"[HP:0000271, HP:0000163, HP:0000455, HP:000049...",2
8,10144,Coffin-Siris syndrome (CSS),"[HP:0000271, HP:0000163, HP:0000455, HP:000021...",2
...,...,...,...,...
84,7997,White-Sutton syndrome (WHSUS),"[HP:0000271, HP:0000455, HP:0000163, HP:000030...",2
85,5008,Williams-Beuren syndrome (WBS),"[HP:0000271, HP:0000163, HP:0000178, HP:000027...",2
86,4991,Williams-Beuren syndrome (WBS),"[HP:0000232, HP:0000280, HP:0000271, HP:000015...",2
87,4982,Williams-Beuren syndrome (WBS),"[HP:0000271, HP:0000163, HP:0000492, HP:000017...",2


In [163]:
image_ids, face_meshes = extract_face_meshes(list(images.values()))
face_mesh_dict = {}
for image_id, face_mesh in zip(image_ids, face_meshes):
    face_mesh_dict[int(image_id)] = torch.tensor(face_mesh[:dimension, :], dtype=torch.float).reshape((dimension, -1))

df_meta = df_gmdb[df_gmdb['image_id'].isin(face_mesh_dict.keys())]
meta_data = {'age': df_meta['age'].tolist(), 'gender': df_meta['gender'].tolist(),
             'ethnicity': df_meta['ethnicity'].tolist()}
meta_data['age'] = torch.stack(
    [torch.tensor(a, dtype=torch.float) if not math.isnan(a) else torch.tensor(-1, dtype=torch.float) for a in
     meta_data['age']])
meta_data['gender'] = torch.stack([torch.tensor(gender_mapping[g], dtype=torch.float) for g in meta_data['gender']])
meta_data['ethnicity'] = torch.stack(
    [torch.tensor(ethnicity_mapping[e], dtype=torch.float) for e in meta_data['ethnicity']])
{key: value.shape for key, value in meta_data.items()}

result = hpo_models.predict(torch.stack(list(face_mesh_dict.values())), use_metric='matthews_corrcoef', **meta_data)

print('Classification completed!')

Face Mesh Extraction: 100%|██████████| 75/75 [00:00<00:00, 83.63it/s]
2026-07-04 12:03:06.911 | WARNING  | lib.hpo_tree.hpo_model:predict:399 - HP:0410030 (Cleft lip) is not trained yet.
2026-07-04 12:03:42.222 | WARNING  | lib.hpo_tree.hpo_model:predict:399 - HP:0004493 (Craniofacial hyperostosis) is not trained yet.


Classification completed!


In [168]:
all_results = {}
for idx, test_sample_row in enumerate(df.iterrows()):
    test_sample_row = test_sample_row[1]
    disorder = test_sample_row['disorder_name']
    if disorder not in all_results:
        all_results[disorder] = []
    image_result = {}
    for key, value in result.items():
        image_result[key.id] = 1 if np.mean(value[idx]) > 0.5 else 0
    all_results[disorder].append(image_result)
all_results = {d: pd.DataFrame(ar).transpose() for d, ar in all_results.items()}

test_gt = {}
for disorder, data_df in all_results.items():
    rows = df[df['disorder_name'] == disorder]
    classified_hpos = data_df.index.tolist()
    gt = {h: [] for h in classified_hpos}
    for idx, r in rows.iterrows():
        for h in classified_hpos:
            if h in r['hpos']:
                gt[h].append(1)
            else:
                gt[h].append(0)
    test_gt[disorder] = pd.DataFrame(gt).transpose()

all_test_results_scores = {d: {id: {} for id in r.index} for d, r in test_gt.items()}
overview_test_results = {d: {id: {} for id in r.index} for d, r in test_gt.items()}
for disorder in test_gt.keys():
    gt_df = test_gt[disorder]
    pred_df = all_results[disorder]
    hpo_indices = gt_df.index.tolist()
    for id in hpo_indices:
        gt = gt_df.loc[id].values
        pred = pred_df.loc[id].values
        auroc = roc_auc_score(gt, pred, labels=[0, 1])
        f1 = f1_score(gt, pred, labels=[0, 1])
        p = precision_score(gt, pred, labels=[0, 1])
        r = recall_score(gt, pred, labels=[0, 1])
        matrix = confusion_matrix(gt, pred, labels=[0, 1])
        FP = matrix.sum(axis=0) - np.diag(matrix)
        FN = matrix.sum(axis=1) - np.diag(matrix)
        TP = np.diag(matrix)
        TN = matrix.sum() - (FP + FN + TP)
        # We only look into the positive class
        FP = FP[1]
        FN = FN[1]
        TP = TP[1]
        TN = TN[1]

        # F1-Score
        F1 = (2*TP)/(2*TP+FP+FN)
        # Sensitivity, hit rate, recall, or true positive rate
        TPR = TP/(TP+FN)
        # Specificity or true negative rate
        TNR = TN/(TN+FP)
        # Precision or positive predictive value
        PPV = TP/(TP+FP)
        # Negative predictive value
        NPV = TN/(TN+FN)
        # Fall out or false positive rate
        FPR = FP/(FP+TN)
        # False negative rate
        FNR = FN/(TP+FN)
        # False discovery rate
        FDR = FP/(TP+FP)
        # Overall accuracy
        ACC = (TP+TN)/(TP+FP+FN+TN)
        # Prevalence
        prev = (TP + FN) / (TP + FN + TN + FP)  # prevalence
        # Detection Prevalence
        prev_det = (TP + FP) / (TP + FN + TN + FP)  # detection prevalence

        all_test_results_scores[disorder][id] = {
            'AUROC': auroc if not np.isnan(auroc) else 'N/A',
            'F1': F1 if not np.isnan(F1) else 'N/A',
            'Prec': PPV if not np.isnan(PPV) else 'N/A',
            'Reca': TPR if not np.isnan(TPR) else 'N/A',
            # 'Spec': TNR if not np.isnan(TNR) else 'N/A',
            'N': len(gt),
            'Prev': prev,
            'Det. P': prev_det,
        }
        overview_test_results[disorder][id] = f1 if prev > 0 else -1

overview_test_results_df = pd.DataFrame(overview_test_results)
left_column = {}
right_column = {}
for c in overview_test_results_df.columns:
    acronym = c[c.find('(') + 1:c.find(')')]
    short_c = c[:5]
    short_c_lower = short_c.lower()
    if len([d for d in train_val_disorders if short_c_lower in d.lower()]) > 0:
        right_column[c] = acronym
    else:
        left_column[c] = acronym
all_columns = {}
all_columns.update(left_column)
all_columns.update(right_column)

for disorder, data in all_test_results_scores.items():
    all_test_results_scores_df = pd.DataFrame(data).transpose()
    all_test_results_scores_df['N'] = all_test_results_scores_df['N'].astype(int)
    all_test_results_scores_df = all_test_results_scores_df[all_test_results_scores_df['Prev'] > 0]
    all_test_results_scores_df.index = [all_hpos[i].name for i in all_test_results_scores_df.index.tolist()]
    use_longtable = len(all_test_results_scores_df) > 40
    disorder_name = disorder[:disorder.find('(') - 1]
    latex = all_test_results_scores_df.to_latex(float_format='%.2f', longtable=True,
                                                caption=f'Prediction results on the test set for {disorder_name}. The HPO terms in bold letters are leafs of the human phenotype ontology. F1: F1-Score, Prec: Precision, Reca: Recall, N: Samples, Prev: Prevalence, Det. P: Detection Prevalence.',
                                                label=f'tab:pred_result_{all_columns[disorder]}',
                                                column_format='r' + 'c' * len(all_test_results_scores_df.columns))

    for lh in leaf_hpos:
        latex = latex.replace(f'{lh.name} ', f'\\textbf{{{lh.name}}} ')
    latex = latex.replace(' ± ', '$\\pm$')

    disorder_name = disorder.replace(";", "_").replace("/", "_").replace(",", "_").replace(" ", "")
    with open(os.path.join(publication_dir, f'test_set_results_{all_columns[disorder]}.tex'), 'w') as f:
        f.write(latex)

overview_test_results_df = overview_test_results_df[all_columns.keys()]
overview_test_results_df.rename(columns=all_columns, inplace=True)
overview_test_results_df = overview_test_results_df[(overview_test_results_df != -1).any(axis=1)]
overview_test_results_df.index = [all_hpos[i].name for i in overview_test_results_df.index.tolist()]

print('All tables created and saved!')

All tables created and saved!


In [169]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np
import os

new_disorders_results_df = overview_test_results_df[all_columns.values()].copy()
# new_disorders_results_df = new_disorders_results_df[(new_disorders_results_df != 0).any(axis=1)]

font_size = 13

leaf_rows_idx = [h for h in new_disorders_results_df.index.tolist() if h in leaf_hpos]
parent_rows_idx = [h for h in new_disorders_results_df.index.tolist() if h not in leaf_hpos]

sorted_leaf_rows_idx = (
    new_disorders_results_df.loc[leaf_rows_idx]
    .mean(axis=1)
    .sort_values(ascending=True)
    .index.tolist()
)
sorted_parent_rows_idx = (
    new_disorders_results_df.loc[parent_rows_idx]
    .mean(axis=1)
    .sort_values(ascending=False)
    .index.tolist()
)

new_disorders_results_df = new_disorders_results_df.reindex(
    sorted_parent_rows_idx + sorted_leaf_rows_idx
)
new_disorders_results_df = new_disorders_results_df[(new_disorders_results_df != -1).any(axis=1)]

cols = new_disorders_results_df.columns.tolist()
n_cols = len(cols)

cmap = plt.get_cmap("tab20")
colors = cmap.colors[:n_cols]

metric_values = []
for hpo_name in new_disorders_results_df.index:
    raw = str(df_best_model_hpos.loc[hpo_name]["F1-Score"])
    mv = float(raw.replace("±", " ").split()[0])
    metric_values.append(mv)
metric_values = np.array(metric_values)

x = np.arange(len(new_disorders_results_df.index))
global_max = max(new_disorders_results_df.values.max(), metric_values.max()) * 1.05

leaf_names = set([h.name for h in leaf_hpos])

fig, axes = plt.subplots(
    n_cols,
    1,
    figsize=(15, max(1.1 * n_cols, 6)),
    sharex=True,
    # sharey=True
)

if n_cols == 1:
    axes = [axes]

bar_width = 0.9

error_result = {}

for j, (ax, col) in enumerate(zip(axes, cols)):
    vals = new_disorders_results_df[col].values

    # optional: skip disorders with no valid bars
    valid_mask = vals != -1
    x_valid = x[valid_mask]
    vals_valid = vals[valid_mask]
    metric_valid = metric_values[valid_mask]

    mv_copy = metric_values.copy()
    vals_copy = vals.copy()
    mv_copy[~valid_mask] = 0
    vals_copy[~valid_mask] = 0
    f1_diff = mv_copy - vals_copy
    f1_diff[~valid_mask] = np.nan

    error_result.update({col: f1_diff})

    ax.bar(
        x_valid,
        vals_valid,
        width=bar_width,
        color=colors[j % len(colors)],
        alpha=0.85,
        zorder=2,
    )

    for xi, mv in zip(x_valid, metric_valid):
        ax.hlines(
            mv,
            xi - bar_width / 2,
            xi + bar_width / 2,
            colors="black",
            linestyles="solid",
            linewidth=2,
            alpha=0.9,
        )

    ax.set_ylim(0, 1)
    ax.set_ylabel("F1", fontsize=font_size)
    ax.set_xlim(-1, max(x) + 1)
    ax.grid(axis="y", alpha=0.25, zorder=1)
    ax.grid(axis="x", alpha=0.25, zorder=1)

    # only show x labels on bottom subplot
    if j < n_cols - 1:
        ax.tick_params(axis="x", labelbottom=False)

# bottom x-axis labels only
axes[-1].set_xticks(x)
axes[-1].set_xticklabels(
    new_disorders_results_df.index.tolist(),
    rotation=90,
    fontsize=font_size
)

for tick in axes[-1].get_xticklabels():
    txt = tick.get_text()
    if txt in leaf_names:
        tick.set_fontweight("bold")

# optional shared legend
handles = [
    Patch(facecolor=colors[i], edgecolor="black", label=cols[i])
    for i in range(n_cols)
]
handles.append(Line2D([0], [0], color="black", lw=1.5, label="Validation"))

fig.legend(
    handles=handles,
    title="Legend",
    loc="lower center",
    bbox_to_anchor=(0.5, 1.01),
    ncol=min(len(handles), 6),
    borderaxespad=0.0,
    fontsize=font_size,
)

plt.margins(x=0.01)
fig.subplots_adjust(
    left=0,
    right=1,
    bottom=0,
    top=1,
    hspace=0.125,  # very tiny vertical spacing
)

plt.savefig(
    os.path.join(publication_dir, "hpo_stacked_disorder_barplots.pdf"),
    format='pdf',
    dpi=300,
    bbox_inches="tight"
)
# plt.show()
plt.close()

df_error_result = pd.DataFrame(columns=new_disorders_results_df.index.tolist(), data=error_result.values(),
                               index=error_result.keys()).transpose()
df_error_result['HPO'] = df_error_result.index.tolist()


def latex_prepare(df_tmp, cols):
    for idx, row in df_error_result[cols].iterrows():
        # find minimum over non-NaN values
        min_val = row.min(skipna=True)

        # mark all entries equal to the minimum as bold (can be ties)
        for col in cols:
            df_tmp[col] = df_tmp[col].astype(str)
            val = row[col]
            if np.isnan(val):
                df_tmp.at[idx, col] = "N/A"
            elif np.isclose(val, min_val, atol=1e-12):
                df_tmp.at[idx, col] = r"\textbf{" + f"{val:.2f}" + "}"
            else:
                df_tmp.at[idx, col] = f"{val:.2f}"


# columns that contain numeric differences (exclude the HPO label)
value_cols = [c for c in df_error_result.columns if c != 'HPO']

# copy for formatting
df_fmt = df_error_result.copy()
df_error_result = df_fmt[['HPO'] + value_cols]
leaf_hpo_names = [lh.name for lh in leaf_hpos]
hpo_mask_is_in = df_error_result['HPO'].isin(leaf_hpo_names)
df_error_result.loc[hpo_mask_is_in, 'HPO'] = df_error_result.loc[hpo_mask_is_in, 'HPO'].apply(
    lambda x: f'\\textbf{{{x}}}')
df_test_perform_unseen = df_error_result[['HPO'] + list(left_column.values())].copy()
latex_prepare(df_test_perform_unseen, left_column.values())
df_test_perform_unseen.to_latex(
    os.path.join(publication_dir, 'test_set_performance_unseen_appendix.tex'),
    float_format='%.2f',
    longtable=True,
    escape=False,
    index=False,
    caption=(
        'An overview of the mean F1-Score performance difference (test set vs. validation set) of the HPO models with a prevalence greater than zero. A number below zero means the performance was better on the test set, while a number above zero means the performance on the validation set was better. The labels in bold letters are leaf nodes of the Human Phenotype Ontology. The names of the unseen disorders are as follows: Susceptibility to Autism (StA), FBXW7 syndrome (FBXW7S), Floating-Harbor syndrome (FLHS), Mowat-Wilson syndrome (MOWS), Nicolaides-Baraitser syndrome (NCBRS), Ohdo syndrome, SBBYS-variant (SBBYSS), Opitz GBBB syndrome (OGBBBS), Sotos syndrome (SOTOS), Seckel syndrome (SS), and White-Sutton syndrome (WHSUS).'
    ),
    label='tab:test_vs_val_unseen_appendix',
    column_format='r' + 'c' * len(left_column)
)

df_test_perform_seen = df_error_result[['HPO'] + list(right_column.values())].copy()
latex_prepare(df_test_perform_seen, right_column.values())
df_test_perform_seen.to_latex(
    os.path.join(publication_dir, 'test_set_performance_seen_appendix.tex'),
    float_format='%.2f',
    longtable=True,
    escape=False,
    index=False,
    caption=(
        'An overview of the mean F1-Score performance differences (test set vs. validation set) of the HPO models with a prevalence greater than zero. A number below zero means the performance was better on the test set, while a number above zero means the performance on the validation set was better. The labels in bold letters are leaf nodes of the Human Phenotype Ontology. The names of the seen disorders are as follows: Coffin-Siris syndrome (CSS), Cornelia de Lange syndrome (CdLS), Hyperphosphatasia with mental retardation syndrome (HPMRS), KBG syndrome (KBGS), Noonan syndrome (NS), Ogden syndrome (OGDNS), and Williams-Beuren syndrome (WBS).'
    ),
    label='tab:test_vs_val_seen_appendix',
    column_format='r' + 'c' * len(right_column)
)

column_name = 'Aggregated F1 Differences'
df_f1_diffs = pd.DataFrame(df_error_result[all_columns.values()].sum(axis=0), columns=[column_name])
hpos_per_disorder = np.asarray([len(df_error_result[df_error_result[s].notnull()]) for s in df_f1_diffs.index.tolist()])
df_f1_diffs[column_name] /= hpos_per_disorder
df_f1_diffs = pd.concat([df_f1_diffs.loc[list(left_column.values())].sort_values(by=column_name, ascending=True),
                         df_f1_diffs.loc[list(right_column.values())].sort_values(by=column_name, ascending=True)],
                        ignore_index=False)
df_f1_diffs['Syndrome'] = [[k for k, v in all_columns.items() if v == d][0] for d in df_f1_diffs.index.tolist()]
df_f1_diffs = df_f1_diffs[['Syndrome', column_name]]
df_f1_diffs.to_latex(
    os.path.join(publication_dir, 'test_set_performance.tex'),
    float_format='%.2f',
    escape=False,
    index=False,
    position='h',
    caption=(
        'An overview of the aggregated mean F1-Score performance differences (test set vs. validation set) of the HPO models with a prevalence greater than zero. The difference value is normalized in the range of -1 to 1, according to the amount of available HPO term per disorder. Above the mid-section-line are the unseen disorders and below the seen disorders.'
    ),
    label='tab:test_vs_val_all',
    column_format='r' + 'c' * len(all_columns)
)

In [170]:
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# Create figure
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.barh(
    df_f1_diffs["Syndrome"],
    df_f1_diffs[column_name],
    edgecolor="black",
    linewidth=0.4,
    zorder=3,
)

# Add values to bars
ax.bar_label(
    bars,
    fmt="%.2f",
    padding=3,
    fontsize=8
)

# Labels and axis styling
ax.set_xlabel("Aggregated Mean F1 Differences by Syndrome")
ax.set_xlim(-1, 1)
ax.invert_yaxis()

# Grid
ax.grid(axis="y", alpha=0.25, zorder=1)
ax.grid(axis="x", alpha=0.25, zorder=1)

# Center line at 0
ax.axvline(0, color="gray", linewidth=1, alpha=0.8, zorder=2)

# Top semantic labels
ax_top = ax.secondary_xaxis("top")
ax_top.set_xlim(ax.get_xlim())

x_min, x_max = ax.get_xlim()
ticks = [x_min, 0.0, x_max]
labels = ["better than validation", "equal", "worse than validation"]

ax_top.set_xticks(ticks)
ax_top.set_xticklabels(labels)
ax_top.tick_params(axis="x", labelsize=9)

# Clean layout
plt.tight_layout()

# Show plot
plt.show()

# Optional: save figure
fig.savefig(os.path.join(publication_dir, "test_set_syndrome_comparison.pdf"), format='pdf', dpi=300, bbox_inches="tight")

In [171]:
r = {c: 0 for c in new_disorders_results_df.columns.tolist()}
for col in r.keys():
    tmp_df = new_disorders_results_df[col]
    tmp_df = tmp_df[tmp_df >= 0]
    r[col] = tmp_df.mean().item()
r

{'StA': 0.696969696969697,
 'FBXW7S': 0.43054673721340386,
 'FLHS': 0.527109440267335,
 'MOWS': 0.6234737484737484,
 'NCBRS': 0.5635944700460829,
 'SBBYSS': 0.6906084656084657,
 'OGBBBS': 0.7470238095238095,
 'SOTOS': 0.7285714285714285,
 'SS': 0.717948717948718,
 'WHSUS': 0.4607142857142857,
 'CSS': 0.700866485903579,
 'CdLS': 0.7563492063492063,
 'HPMRS': 0.7793650793650794,
 'KBGS': 0.6791666666666666,
 'NS': 0.6756235827664399,
 'OGDNS': 0.6674418604651162,
 'WBS': 0.803265306122449}